# Bitcoin transaction classification

**Objective:** rank transactions by illicit probability using confirmed labels, all 165 anonymized features, and the directed transaction graph.

The analysis compares logistic regression, HGB and CatBoost, followed by probability blending, subtype weighting and graph-based feature engineering. The two selected strategies are **HGB–CatBoost Graph Ensemble** (best confirmed Kaggle public ROC-AUC: **0.95132**) and **HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting** (slightly higher local ROC-AUC; Kaggle score not yet reported).


## 1. Data and working environment

The analysis uses four competition files: `train.csv`, `test.csv`, `txs_edgelist.csv` and `sample_submission.csv`, stored in `dataset/`. Python, NumPy, pandas and SciPy support data preparation and graph calculations; scikit-learn and CatBoost provide the classifiers, preprocessing and evaluation tools.


In [1]:
from pathlib import Path
from collections import deque
from io import BytesIO
import base64
import hashlib
import warnings

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.csgraph import connected_components
from sklearn.cluster import KMeans
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from catboost import CatBoostClassifier
from threadpoolctl import threadpool_limits

DATA_DIR = Path('dataset')
FEATURES = [f'feat_{i}' for i in range(1, 166)]
SEEDS = [47, 48, 49]
USE_RECORDED_RESULTS = True
REBUILD_SUBMISSIONS = False

required = ['train.csv', 'test.csv', 'txs_edgelist.csv', 'sample_submission.csv']
missing = [name for name in required if not (DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f'Missing data files: {missing}; folder: {DATA_DIR.resolve()}')
train = pd.read_csv(DATA_DIR / 'train.csv', dtype={'label': 'Int64'},
                    na_values={'label': ['none', 'None', 'unknown', 'Unknown']})
test = pd.read_csv(DATA_DIR / 'test.csv')
edges = pd.read_csv(DATA_DIR / 'txs_edgelist.csv')
sample = pd.read_csv(DATA_DIR / 'sample_submission.csv')
pd.set_option('display.precision', 6)
print('Four competition files loaded. Recorded results:', USE_RECORDED_RESULTS)


Four competition files loaded. Recorded results: True


## 2. Exploratory data analysis

Each node is a transaction, identified by `txId`. The 165 numeric features are anonymized and pre-normalized. `label=1` means illicit, `label=0` licit, and a missing label means unknown. A directed edge `(txId1, txId2)` indicates that the latter transaction spends an output created by the former.

The initial checks cover dataset dimensions, schema consistency, missing values, duplicates, label availability and time-step coverage. The test `index` is used solely to match the sample submission.


In [2]:
quality = []
for name, frame in [('train', train), ('test', test)]:
    quality.append(dict(dataset=name, rows=len(frame), columns=frame.shape[1],
        features=sum(c.startswith('feat_') for c in frame.columns),
        missing_feature_cells=int(frame[FEATURES].isna().sum().sum()),
        duplicate_rows=int(frame.duplicated().sum()),
        duplicate_txIds=int(frame.txId.duplicated().sum())))
display(pd.DataFrame(quality).set_index('dataset'))
missing_columns = train.isna().sum()
display(missing_columns[missing_columns.gt(0)].rename('missing_training_values').to_frame())
label_counts = pd.Series({'licit': int(train.label.eq(0).sum()),
    'illicit': int(train.label.eq(1).sum()), 'unknown': int(train.label.isna().sum())})
display(pd.DataFrame({'rows': label_counts, 'percent_of_train': 100 * label_counts / len(train)}))
print('Train/test transaction overlap:', len(set(train.txId) & set(test.txId)))


,rows,columns,features,missing_feature_cells,duplicate_rows,duplicate_txIds
dataset,,,,,,
train,141772,168,165,0,0,0
test,15329,168,165,0,0,0


,missing_training_values
label,110537


,rows,percent_of_train
licit,27591,19.461530
illicit,3644,2.570324
unknown,110537,77.968146


Train/test transaction overlap: 0


**Finding and implication.** Training contains 141,772 transactions, of which 27,591 are confirmed licit, 3,644 confirmed illicit, and 110,537 unknown. Only **31,235** rows are eligible for supervised fitting. Missing target values represent unknown status and are excluded from supervised fitting. There are no missing feature values, duplicate transaction IDs or shared train/test IDs in the supplied files. Median imputation is retained as a training-fitted safeguard. The illicit share is approximately 11.7% of confirmed labels, so ROC-AUC is supplemented with precision, recall and F1.


In [3]:
def time_summary(name, frame):
    steps = sorted(frame.time_step.unique())
    return dict(dataset=name, first=int(min(steps)), last=int(max(steps)),
        distinct_steps=len(steps), missing_steps=sorted(set(range(int(min(steps)), int(max(steps))+1))-set(steps)))
display(pd.DataFrame([time_summary('train', train), time_summary('test', test)]))
assert edges.columns.tolist() == ['txId1', 'txId2'] and not edges.isna().any().any()
edge_duplicates = int(edges.duplicated().sum())
edges = edges.drop_duplicates().reset_index(drop=True)
node_ids = pd.Index(np.union1d(edges.txId1, edges.txId2))
src = node_ids.get_indexer(edges.txId1)
dst = node_ids.get_indexer(edges.txId2)
adj = sparse.csr_matrix((np.ones(len(edges), dtype=np.float32), (src, dst)), shape=(len(node_ids), len(node_ids)))
adj.sum_duplicates(); adj.data[:] = 1
component_count, components = connected_components(adj, directed=False)
display(pd.DataFrame([dict(nodes=len(node_ids), edges=int(adj.nnz),
    duplicate_edges_removed=edge_duplicates, self_loops=int(adj.diagonal().sum()),
    weak_components=component_count)]))
metadata = pd.concat([train[['txId', 'time_step']], test[['txId', 'time_step']]])
metadata = metadata.assign(component=components[node_ids.get_indexer(metadata.txId)])
assert metadata.groupby('component').time_step.nunique().max() == 1
component_steps = metadata.groupby('component').time_step.first().reindex(range(component_count))
assert component_steps.notna().all()
node_steps = component_steps.to_numpy()[components]
assert np.array_equal(node_steps[src], node_steps[dst])
test_period_nodes = int((node_steps >= test.time_step.min()).sum())
print(f'Test attributes cover {len(test):,}/{test_period_nodes:,} graph nodes ({len(test)/test_period_nodes:.1%}).')


,dataset,first,last,distinct_steps,missing_steps
0,train,1,35,35,[]
1,test,36,49,14,[]


,nodes,edges,duplicate_edges_removed,self_loops,weak_components
0,203769,234355,0,0,49


Test attributes cover 15,329/61,997 graph nodes (24.7%).


**Finding and implication.** Training covers steps 1–35 and test covers 36–49, with no overlap. This creates a genuine future-period distribution gap. The graph has 203,769 nodes and 234,355 directed edges; its 49 weak components each lie within a single period. An edge gives causal transaction order within a snapshot, but the coarse time step cannot order its endpoints.

Only about **24.7%** of nodes in test-period graph components have supplied numeric attributes. This motivates using complete-graph structural descriptors rather than assuming every neighbor has observable features. Graph topology is available for nodes with unknown labels or unavailable attributes. No test labels are used.


## 3. Data preparation

Numeric columns and transaction identifiers are validated, identical duplicate rows are removed, and nonfinite feature values are replaced with missing values. Confirmed and unknown transactions form separate cohorts. Median imputers are fitted within each fitting subset.

**All 165 original features** are retained without feature selection. Identifiers and `time_step` are excluded from classifier inputs. Time is used for HGB recency weights and the label-free within-period subtype representation.


In [4]:
def clean(frame, training=False):
    assert frame.columns.is_unique
    assert {c for c in frame if c.startswith('feat_')} == set(FEATURES)
    result = frame.drop_duplicates().copy()
    assert result.txId.notna().all() and result.txId.is_unique
    steps = pd.to_numeric(result.time_step, errors='raise')
    assert steps.notna().all() and np.isfinite(steps).all() and steps.mod(1).eq(0).all()
    result['time_step'] = steps.astype('int64')
    values = result[FEATURES].apply(pd.to_numeric, errors='raise').astype(float)
    result[FEATURES] = values.replace([np.inf, -np.inf], np.nan)
    if training:
        assert result.label.dropna().isin([0, 1]).all()
    return result

train_clean = clean(train, True)
test_clean = clean(test)
confirmed = train_clean.loc[train_clean.label.isin([0, 1])].copy()
unknown = train_clean.loc[train_clean.label.isna()].copy()
y = confirmed.label.to_numpy(dtype=int)
X = confirmed[FEATURES].to_numpy(dtype=float)
times = confirmed.time_step.to_numpy()
assert set(confirmed.txId).isdisjoint(unknown.txId)
assert set(train_clean.txId).isdisjoint(test_clean.txId)
assert len(confirmed) + len(unknown) == len(train_clean)
assert unknown.label.isna().all() and np.isfinite(X).all()
display(pd.DataFrame([{'group': 'confirmed supervised', 'rows': len(confirmed)},
    {'group': 'unknown; no supervised targets', 'rows': len(unknown)},
    {'group': 'test; withheld targets', 'rows': len(test_clean)}]))


,group,rows
0,confirmed supervised,31235
1,unknown; no supervised targets,110537
2,test; withheld targets,15329


## 4. Validation design

**Three-fold stratified random validation is repeated with seeds 47, 48 and 49.** Every model uses the same confirmed-label rows and splits. Performance is summarized by mean AUC, weakest-fold AUC and matched fold changes. Precision, recall and F1 use a probability threshold of 0.5.

**Limitations:** random folds mix periods and can share related graph neighborhoods, so they do not establish generalization to unseen periods. The repeated shuffles overlap, and extensive reuse of these folds adds selection optimism. Period-specific error summaries are descriptive slices rather than independent future-period evaluations.

Imputers, scalers, subtype clusters and fitted numeric transformations exclude held-out transactions. Unknown training rows contribute covariates to a label-free period reference. Complete snapshot topology is used without labels, consistent with prediction after the full snapshot graph is available. Validation labels are used only for scoring.


In [5]:
SPLITS = [(seed, fold, a, b) for seed in SEEDS
    for fold, (a, b) in enumerate(StratifiedKFold(3, shuffle=True, random_state=seed).split(X, y), 1)]
for seed, fold, a, b in SPLITS:
    assert not np.intersect1d(a, b).size and set(y[a]) == set(y[b]) == {0, 1}
for seed in SEEDS:
    heldout = np.concatenate([b for s, _, _, b in SPLITS if s == seed])
    assert np.array_equal(np.sort(heldout), np.arange(len(y)))
print('Matched random-validation folds:', len(SPLITS), '; three out-of-fold passes.')

def metrics(name, seed, fold, truth, probability):
    return dict(model=name, seed=seed, fold=fold,
        auc=float(roc_auc_score(truth, probability)),
        ap=float(average_precision_score(truth, probability)),
        precision=float(precision_score(truth, probability >= .5, zero_division=0)),
        recall=float(recall_score(truth, probability >= .5, zero_division=0)),
        f1=float(f1_score(truth, probability >= .5, zero_division=0)))


Matched random-validation folds: 9 ; three out-of-fold passes.


In [6]:
RECORDED_RESULTS = [{'model': 'Logistic regression', 'seed': 47, 'fold': 1, 'auc': 0.9734116197310717, 'ap': 0.8402335444110487, 'precision': 0.8239669421487603, 'recall': 0.8205761316872427, 'f1': 0.8222680412371134}, {'model': 'Logistic regression', 'seed': 47, 'fold': 2, 'auc': 0.9759525270138635, 'ap': 0.8271449630547717, 'precision': 0.8321048321048321, 'recall': 0.8362139917695474, 'f1': 0.8341543513957307}, {'model': 'Logistic regression', 'seed': 47, 'fold': 3, 'auc': 0.976485599218569, 'ap': 0.85842959654949, 'precision': 0.8429276315789473, 'recall': 0.8443163097199341, 'f1': 0.8436213991769548}, {'model': 'Logistic regression', 'seed': 48, 'fold': 1, 'auc': 0.9715839527203136, 'ap': 0.8309017781745972, 'precision': 0.8068091844813935, 'recall': 0.8386831275720165, 'f1': 0.8224374495560937}, {'model': 'Logistic regression', 'seed': 48, 'fold': 2, 'auc': 0.9786431521103456, 'ap': 0.857339151546032, 'precision': 0.8542713567839196, 'recall': 0.8395061728395061, 'f1': 0.8468244084682441}, {'model': 'Logistic regression', 'seed': 48, 'fold': 3, 'auc': 0.9735472619375382, 'ap': 0.8456440994639512, 'precision': 0.8477331052181352, 'recall': 0.8163097199341022, 'f1': 0.8317247167436005}, {'model': 'Logistic regression', 'seed': 49, 'fold': 1, 'auc': 0.9762819420002318, 'ap': 0.8420812502580837, 'precision': 0.81640625, 'recall': 0.8600823045267489, 'f1': 0.8376753507014028}, {'model': 'Logistic regression', 'seed': 49, 'fold': 2, 'auc': 0.9735574894479369, 'ap': 0.8381228127810085, 'precision': 0.8330536912751678, 'recall': 0.817283950617284, 'f1': 0.8250934773577067}, {'model': 'Logistic regression', 'seed': 49, 'fold': 3, 'auc': 0.9739487788708409, 'ap': 0.8454381662612493, 'precision': 0.8355371900826446, 'recall': 0.8327841845140033, 'f1': 0.8341584158415841}, {'seed': 47, 'fold': 1, 'auc': 0.9985975924337467, 'ap': 0.9931545961312117, 'precision': 0.9872122762148338, 'recall': 0.9530864197530864, 'f1': 0.9698492462311558, 'model': 'HGB'}, {'seed': 47, 'fold': 2, 'auc': 0.9989055296703925, 'ap': 0.9942360428972454, 'precision': 0.9880444064901793, 'recall': 0.9522633744855967, 'f1': 0.9698239731768651, 'model': 'HGB'}, {'seed': 47, 'fold': 3, 'auc': 0.9986102301463176, 'ap': 0.9923181242358026, 'precision': 0.9879101899827288, 'recall': 0.942339373970346, 'f1': 0.9645868465430016, 'model': 'HGB'}, {'seed': 48, 'fold': 1, 'auc': 0.9988930904736784, 'ap': 0.9946610414903314, 'precision': 0.9888698630136986, 'recall': 0.9506172839506173, 'f1': 0.9693663449433487, 'model': 'HGB'}, {'seed': 48, 'fold': 2, 'auc': 0.998858368111627, 'ap': 0.9938274563507757, 'precision': 0.9897084048027445, 'recall': 0.9497942386831276, 'f1': 0.9693406131877362, 'model': 'HGB'}, {'seed': 48, 'fold': 3, 'auc': 0.9978395290062173, 'ap': 0.9899869033554212, 'precision': 0.9845758354755784, 'recall': 0.9464579901153213, 'f1': 0.9651406971860563, 'model': 'HGB'}, {'seed': 49, 'fold': 2, 'auc': 0.998827493846401, 'ap': 0.9926955568433556, 'precision': 0.9862542955326461, 'recall': 0.9448559670781893, 'f1': 0.9651113913408995, 'model': 'HGB'}, {'seed': 49, 'fold': 1, 'auc': 0.9983344899996465, 'ap': 0.9921741536447545, 'precision': 0.9845360824742269, 'recall': 0.9432098765432099, 'f1': 0.9634300126103404, 'model': 'HGB'}, {'seed': 49, 'fold': 3, 'auc': 0.9985432360204844, 'ap': 0.9937486202042759, 'precision': 0.9872665534804754, 'recall': 0.9579901153212521, 'f1': 0.9724080267558528, 'model': 'HGB'}, {'seed': 47, 'fold': 1, 'auc': 0.9981362682678329, 'ap': 0.9929729979537874, 'precision': 0.9948186528497409, 'recall': 0.9481481481481482, 'f1': 0.9709228824273072, 'model': 'CatBoost'}, {'seed': 47, 'fold': 2, 'auc': 0.9988804722957164, 'ap': 0.9943948335168152, 'precision': 0.9898132427843803, 'recall': 0.9596707818930041, 'f1': 0.9745089845382365, 'model': 'CatBoost'}, {'seed': 47, 'fold': 3, 'auc': 0.9987698337990379, 'ap': 0.9929405336783141, 'precision': 0.9922077922077922, 'recall': 0.943986820428336, 'f1': 0.9674968341072182, 'model': 'CatBoost'}, {'seed': 48, 'fold': 1, 'auc': 0.9984669361229351, 'ap': 0.9944516631458732, 'precision': 0.993960310612597, 'recall': 0.9481481481481482, 'f1': 0.970513900589722, 'model': 'CatBoost'}, {'seed': 48, 'fold': 2, 'auc': 0.9990256260875907, 'ap': 0.9946950003913705, 'precision': 0.9940017137960583, 'recall': 0.9547325102880658, 'f1': 0.9739714525608733, 'model': 'CatBoost'}, {'seed': 48, 'fold': 3, 'auc': 0.9974796594907122, 'ap': 0.9895224060748287, 'precision': 0.989628349178911, 'recall': 0.943163097199341, 'f1': 0.9658371994938845, 'model': 'CatBoost'}, {'seed': 49, 'fold': 2, 'auc': 0.9987601968972706, 'ap': 0.9927080041852162, 'precision': 0.9939288811795317, 'recall': 0.9432098765432099, 'f1': 0.9679054054054054, 'model': 'CatBoost'}, {'seed': 49, 'fold': 1, 'auc': 0.9980284320661013, 'ap': 0.9921521007189521, 'precision': 0.9888123924268503, 'recall': 0.945679012345679, 'f1': 0.9667648296171645, 'model': 'CatBoost'}, {'seed': 49, 'fold': 3, 'auc': 0.9985138589171778, 'ap': 0.9943094773094071, 'precision': 0.9931565440547476, 'recall': 0.9563426688632619, 'f1': 0.9744020142677298, 'model': 'CatBoost'}, {'seed': 47, 'fold': 1, 'auc': 0.9986067204773788, 'ap': 0.9934811388490817, 'precision': 0.992274678111588, 'recall': 0.951440329218107, 'f1': 0.9714285714285714, 'model': 'HGB + CatBoost'}, {'seed': 47, 'fold': 2, 'auc': 0.998996899597337, 'ap': 0.9947684664583355, 'precision': 0.989778534923339, 'recall': 0.9563786008230453, 'f1': 0.972791963164504, 'model': 'HGB + CatBoost'}, {'seed': 47, 'fold': 3, 'auc': 0.9988918204292317, 'ap': 0.9934470670146102, 'precision': 0.9896551724137931, 'recall': 0.9456342668863262, 'f1': 0.9671440606571188, 'model': 'HGB + CatBoost'}, {'seed': 48, 'fold': 1, 'auc': 0.9988750133676619, 'ap': 0.995114817329621, 'precision': 0.9939810834049871, 'recall': 0.951440329218107, 'f1': 0.9722455845248108, 'model': 'HGB + CatBoost'}, {'seed': 48, 'fold': 2, 'auc': 0.9990566793340645, 'ap': 0.9947094197526021, 'precision': 0.9948364888123924, 'recall': 0.951440329218107, 'f1': 0.9726546066470341, 'model': 'HGB + CatBoost'}, {'seed': 48, 'fold': 3, 'auc': 0.9979206742976678, 'ap': 0.9904201158098997, 'precision': 0.9879414298018949, 'recall': 0.9448105436573312, 'f1': 0.9658947368421053, 'model': 'HGB + CatBoost'}, {'seed': 49, 'fold': 2, 'auc': 0.9989322873669217, 'ap': 0.9933488657048035, 'precision': 0.9888123924268503, 'recall': 0.945679012345679, 'f1': 0.9667648296171645, 'model': 'HGB + CatBoost'}, {'seed': 49, 'fold': 1, 'auc': 0.9984157474860965, 'ap': 0.9928826957359216, 'precision': 0.9896462467644521, 'recall': 0.9440329218106996, 'f1': 0.966301600673968, 'model': 'HGB + CatBoost'}, {'seed': 49, 'fold': 3, 'auc': 0.9986482054262017, 'ap': 0.9944083536391156, 'precision': 0.9940222032450897, 'recall': 0.9588138385502472, 'f1': 0.9761006289308176, 'model': 'HGB + CatBoost'}, {'seed': 47, 'fold': 1, 'auc': 0.9985963395650128, 'ap': 0.9936435027697229, 'precision': 0.9905741216795202, 'recall': 0.951440329218107, 'f1': 0.9706129303106633, 'model': 'Feature-subtype blend'}, {'seed': 47, 'fold': 2, 'auc': 0.999028937240673, 'ap': 0.9949255848885273, 'precision': 0.9856054191363252, 'recall': 0.9580246913580247, 'f1': 0.9716193656093489, 'model': 'Feature-subtype blend'}, {'seed': 47, 'fold': 3, 'auc': 0.9988337827373335, 'ap': 0.9931494724820783, 'precision': 0.9879518072289156, 'recall': 0.9456342668863262, 'f1': 0.9663299663299664, 'model': 'Feature-subtype blend'}, {'seed': 48, 'fold': 1, 'auc': 0.99880977470288, 'ap': 0.9952510953886529, 'precision': 0.9922879177377892, 'recall': 0.9530864197530864, 'f1': 0.9722921914357683, 'model': 'Feature-subtype blend'}, {'seed': 48, 'fold': 2, 'auc': 0.9990801258775115, 'ap': 0.9944651248178091, 'precision': 0.9923142613151152, 'recall': 0.9563786008230453, 'f1': 0.9740150880134115, 'model': 'Feature-subtype blend'}, {'seed': 48, 'fold': 3, 'auc': 0.998154616352048, 'ap': 0.9911077026501186, 'precision': 0.9879414298018949, 'recall': 0.9448105436573312, 'f1': 0.9658947368421053, 'model': 'Feature-subtype blend'}, {'seed': 49, 'fold': 1, 'auc': 0.9984226382641325, 'ap': 0.9930285458267661, 'precision': 0.9888507718696398, 'recall': 0.9489711934156378, 'f1': 0.9685006299874003, 'model': 'Feature-subtype blend'}, {'seed': 49, 'fold': 2, 'auc': 0.9990623172433666, 'ap': 0.9938993107888051, 'precision': 0.9897260273972602, 'recall': 0.951440329218107, 'f1': 0.9702056231640789, 'model': 'Feature-subtype blend'}, {'seed': 49, 'fold': 3, 'auc': 0.9987212899271107, 'ap': 0.9944694859711276, 'precision': 0.9931914893617021, 'recall': 0.9612850082372323, 'f1': 0.9769778149853495, 'model': 'Feature-subtype blend'}, {'seed': 47, 'fold': 1, 'auc': 0.9986119109335617, 'ap': 0.9936846085467698, 'precision': 0.9914163090128756, 'recall': 0.9506172839506173, 'f1': 0.9705882352941176, 'model': 'Graph-weighted blend'}, {'seed': 47, 'fold': 2, 'auc': 0.9990423608342495, 'ap': 0.9949706772738524, 'precision': 0.9872881355932204, 'recall': 0.9588477366255144, 'f1': 0.9728601252609603, 'model': 'Graph-weighted blend'}, {'seed': 47, 'fold': 3, 'auc': 0.9988267967188641, 'ap': 0.9931379211649786, 'precision': 0.9879518072289156, 'recall': 0.9456342668863262, 'f1': 0.9663299663299664, 'model': 'Graph-weighted blend'}, {'seed': 48, 'fold': 1, 'auc': 0.9988550569585447, 'ap': 0.9953118276099923, 'precision': 0.9931330472103004, 'recall': 0.9522633744855967, 'f1': 0.9722689075630252, 'model': 'Graph-weighted blend'}, {'seed': 48, 'fold': 2, 'auc': 0.9990759198181909, 'ap': 0.9946398244088855, 'precision': 0.9940119760479041, 'recall': 0.9563786008230453, 'f1': 0.9748322147651006, 'model': 'Graph-weighted blend'}, {'seed': 48, 'fold': 3, 'auc': 0.9981210297247921, 'ap': 0.9909395544017676, 'precision': 0.9879310344827587, 'recall': 0.943986820428336, 'f1': 0.9654591406908172, 'model': 'Graph-weighted blend'}, {'seed': 49, 'fold': 1, 'auc': 0.9984364198202045, 'ap': 0.9930328887848303, 'precision': 0.9888412017167382, 'recall': 0.9481481481481482, 'f1': 0.9680672268907563, 'model': 'Graph-weighted blend'}, {'seed': 49, 'fold': 2, 'auc': 0.9990630331683574, 'ap': 0.9938997245848583, 'precision': 0.9897172236503856, 'recall': 0.9506172839506173, 'f1': 0.9697732997481109, 'model': 'Graph-weighted blend'}, {'seed': 49, 'fold': 3, 'auc': 0.9987073178901722, 'ap': 0.9943937735405753, 'precision': 0.9915038232795242, 'recall': 0.9612850082372323, 'f1': 0.9761606022584692, 'model': 'Graph-weighted blend'}, {'seed': 47, 'fold': 1, 'auc': 0.9986876200013335, 'ap': 0.9937935992859133, 'precision': 0.9931330472103004, 'recall': 0.9522633744855967, 'f1': 0.9722689075630252, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 47, 'fold': 2, 'auc': 0.9990343961687274, 'ap': 0.9949519465206795, 'precision': 0.9864291772688719, 'recall': 0.9572016460905349, 'f1': 0.9715956558061821, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 47, 'fold': 3, 'auc': 0.9989106289404951, 'ap': 0.9935372098820329, 'precision': 0.9905254091300603, 'recall': 0.9472817133443163, 'f1': 0.968421052631579, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 48, 'fold': 3, 'auc': 0.9980843979099983, 'ap': 0.9907914901842728, 'precision': 0.9871244635193133, 'recall': 0.9472817133443163, 'f1': 0.9667927700714586, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 48, 'fold': 2, 'auc': 0.9990923860929781, 'ap': 0.9946728516387142, 'precision': 0.9923076923076923, 'recall': 0.9555555555555556, 'f1': 0.9735849056603774, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 48, 'fold': 1, 'auc': 0.998838322211886, 'ap': 0.9952291269030127, 'precision': 0.9948497854077253, 'recall': 0.9539094650205762, 'f1': 0.9739495798319328, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 49, 'fold': 1, 'auc': 0.9985236731784519, 'ap': 0.993265838107542, 'precision': 0.9905498281786942, 'recall': 0.9489711934156378, 'f1': 0.9693148381672971, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 49, 'fold': 3, 'auc': 0.9988333349156366, 'ap': 0.9946174275131032, 'precision': 0.9923404255319149, 'recall': 0.9604612850082372, 'f1': 0.9761406446211804, 'model': 'HGB–CatBoost Graph Ensemble'}, {'seed': 49, 'fold': 2, 'auc': 0.9990784255556585, 'ap': 0.9940152993653115, 'precision': 0.988860325621251, 'recall': 0.9497942386831276, 'f1': 0.9689336691855583, 'model': 'HGB–CatBoost Graph Ensemble'}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 47, 'fold': 1, 'auc': 0.9987075764104505, 'ap': 0.9939359716698508, 'precision': 0.9905741216795202, 'recall': 0.951440329218107, 'f1': 0.9706129303106633}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 47, 'fold': 2, 'auc': 0.9990740405150902, 'ap': 0.9951413316980106, 'precision': 0.9865092748735245, 'recall': 0.9629629629629629, 'f1': 0.9745939192003332}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 47, 'fold': 3, 'auc': 0.9988926265082858, 'ap': 0.993455956897282, 'precision': 0.9913867355727821, 'recall': 0.9481054365733114, 'f1': 0.9692631578947368}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 48, 'fold': 1, 'auc': 0.9989117940140616, 'ap': 0.9952697066529808, 'precision': 0.9948364888123924, 'recall': 0.951440329218107, 'f1': 0.9726546066470341}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 48, 'fold': 2, 'auc': 0.999119859714498, 'ap': 0.9948025826114331, 'precision': 0.9914675767918089, 'recall': 0.9563786008230453, 'f1': 0.9736070381231672}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 48, 'fold': 3, 'auc': 0.9981111776474637, 'ap': 0.9909231850252642, 'precision': 0.9862778730703259, 'recall': 0.9472817133443163, 'f1': 0.9663865546218487}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 49, 'fold': 1, 'auc': 0.9985741458902997, 'ap': 0.9934096441062276, 'precision': 0.9880341880341881, 'recall': 0.951440329218107, 'f1': 0.969392033542977}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 49, 'fold': 2, 'auc': 0.9990210620657748, 'ap': 0.9937328283304785, 'precision': 0.9888698630136986, 'recall': 0.9506172839506173, 'f1': 0.9693663449433487}, {'model': 'HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting', 'seed': 49, 'fold': 3, 'auc': 0.9987740433229875, 'ap': 0.9945885499372832, 'precision': 0.989821882951654, 'recall': 0.9612850082372323, 'f1': 0.9753447555369829}]

with np.load(BytesIO(base64.b64decode('...')), allow_pickle=False) as archive:
    FINAL_OOF = {name: archive[name].copy() for name in archive.files}
assert np.array_equal(FINAL_OOF["txId"], confirmed.txId.to_numpy())
assert np.array_equal(FINAL_OOF["label"], y)


In [7]:
def summarize(rows):
    return pd.DataFrame(rows).groupby('model', sort=False).agg(
        folds=('auc','size'), mean_auc=('auc','mean'), worst_auc=('auc','min'),
        mean_AP=('ap','mean'), precision=('precision','mean'), recall=('recall','mean'), F1=('f1','mean'))

def show_stage(name):
    rows = [r for r in RECORDED_RESULTS if r['model'] == name] if USE_RECORDED_RESULTS else evaluate_stage(name)
    assert len(rows) == 9
    display(summarize(rows))
    return rows


## 5. Logistic regression baseline

Logistic regression provides a linear benchmark. The pipeline fits a median imputer and standard scaler on fitting rows only, followed by logistic regression (`C=1`, `lbfgs`, at most 2,000 iterations). All 165 original columns are used. Evaluation uses the same nine random folds as the boosting experiments.


In [8]:
def logistic_baseline():
    rows = []
    for seed, fold, a, b in SPLITS:
        model = Pipeline([('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('scaler', StandardScaler()),
            ('classifier', LogisticRegression(C=1., solver='lbfgs', max_iter=2000, random_state=seed))])
        with threadpool_limits(limits=2):
            model.fit(X[a], y[a])
            probability = model.predict_proba(X[b])[:, 1]
        if model.named_steps['classifier'].n_iter_.max() >= 2000:
            raise RuntimeError('Baseline did not converge; do not report this fit.')
        rows.append(metrics('Logistic regression', seed, fold, y[b], probability))
    return rows

baseline_results = [r for r in RECORDED_RESULTS if r['model'] == 'Logistic regression'] if USE_RECORDED_RESULTS else logistic_baseline()
display(summarize(baseline_results))


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
Logistic regression,9,0.974824,0.971584,0.842815,0.832534,0.833973,0.833106


**Result and interpretation.** Mean random-validation ROC-AUC is **0.974824**. This provides the reference for assessing nonlinear trees and subsequent techniques on identical splits. Local validation and Kaggle public scores measure performance on different datasets.


## 6. Model development

The analysis compares HGB and CatBoost individually before examining their probability blend, subtype weighting and graph-based features. The resulting strategies use both learner families. Each experiment is accompanied by its rationale, validation results and interpretation.

Both final strategies train only on confirmed labels. Unknown transactions contribute feature-distribution and graph-structure information without receiving pseudo-labels.


### 6.1 Graph representation and model settings

The graph representation comprises 12 label-free descriptors: log in/out-degree; source/sink flags; longest depth from roots and to sinks; relative depth; log degree product; and shared-successor/shared-predecessor neighbor counts and hub-normalized masses. Degree describes observed transaction connectivity rather than exact Bitcoin input/output counts. Acyclicity is verified before computing depth.

Earlier stages use 165 numeric inputs. Both final graph ensembles append these descriptors, giving **177 classifier inputs**. Neighbor-attribute aggregates are excluded because attribute availability differs sharply between source and test graph snapshots.


In [9]:
def build_graph_features(adjacency):
    indegree = np.asarray(adjacency.sum(axis=0)).ravel().astype(np.int32)
    outdegree = np.asarray(adjacency.sum(axis=1)).ravel().astype(np.int32)
    remaining = indegree.copy()
    queue = deque(np.flatnonzero(remaining == 0).tolist())
    order = []; depth_in = np.zeros(len(indegree), dtype=np.int32)
    while queue:
        u = queue.popleft(); order.append(u)
        for v in adjacency.indices[adjacency.indptr[u]:adjacency.indptr[u+1]]:
            depth_in[v] = max(depth_in[v], depth_in[u]+1)
            remaining[v] -= 1
            if remaining[v] == 0: queue.append(int(v))
    assert len(order) == len(indegree), 'The graph must be acyclic.'
    depth_out = np.zeros(len(indegree), dtype=np.int32)
    for u in reversed(order):
        neighbors = adjacency.indices[adjacency.indptr[u]:adjacency.indptr[u+1]]
        if len(neighbors): depth_out[u] = depth_out[neighbors].max()+1
    blocks = [np.log1p(indegree), np.log1p(outdegree), (indegree==0).astype(float),
        (outdegree==0).astype(float), np.log1p(depth_in), np.log1p(depth_out),
        depth_in / np.maximum(depth_in+depth_out,1), np.log1p(indegree.astype(float)*outdegree)]
    for direction in ['shared_successor', 'shared_predecessor']:
        relation = (adjacency @ sparse.diags(1/np.maximum(indegree,1)) @ adjacency.T).tocsr() if direction=='shared_successor' else (adjacency.T @ sparse.diags(1/np.maximum(outdegree,1)) @ adjacency).tocsr()
        relation.setdiag(0); relation.eliminate_zeros(); relation.sort_indices()
        blocks.extend([np.log1p(np.diff(relation.indptr)), np.log1p(np.asarray(relation.sum(axis=1)).ravel())])
    topology = np.column_stack(blocks).astype(float)
    assert topology.shape == (len(node_ids), 12) and np.isfinite(topology).all()
    return topology, indegree, outdegree

topology, indegree, outdegree = build_graph_features(adj)
known_nodes = node_ids.get_indexer(confirmed.txId)
test_nodes = node_ids.get_indexer(test_clean.txId)
assert (known_nodes >= 0).all() and (test_nodes >= 0).all()
G = topology[known_nodes]; G_test = topology[test_nodes]
print('Final classifier inputs:', X.shape[1], 'original +', G.shape[1], 'graph =', X.shape[1]+G.shape[1])


Final classifier inputs: 165 original + 12 graph = 177


In [10]:
HGB_SETTINGS = dict(max_iter=300, learning_rate=.05, max_leaf_nodes=15,
    min_samples_leaf=100, l2_regularization=5, early_stopping=False)
CAT_SETTINGS = dict(iterations=800, depth=6, learning_rate=.05, l2_leaf_reg=3,
    loss_function='Logloss', eval_metric='AUC', allow_writing_files=False, verbose=False)

def make_model(family, seed, final=False):
    if family == 'HGB':
        return HistGradientBoostingClassifier(**HGB_SETTINGS, random_state=seed)
    return CatBoostClassifier(**CAT_SETTINGS, random_seed=seed, thread_count=1 if final else 4)

display(pd.DataFrame([{'family':'HGB', 'settings':HGB_SETTINGS}, {'family':'CatBoost', 'settings':CAT_SETTINGS}]))


,family,settings
0,HGB,"{'max_iter': 300, 'learning_rate': 0.05, 'max_leaf_nodes': 15, 'min_samples_leaf': 100, 'l2_regularization': 5, 'early_stopping': False}"
1,CatBoost,"{'iterations': 800, 'depth': 6, 'learning_rate': 0.05, 'l2_leaf_reg': 3, 'loss_function': 'Logloss', 'eval_metric': 'AUC', 'allow_writing_files': False, 'verbose': False}"


The fixed configurations above are used in both final strategies following bounded parameter tuning. HGB uses recency weights with a **20-step half-life**, calculated from the fitting subset's latest period. CatBoost has no recency weighting. Neither strategy uses a global class-weight multiplier.

Subtype weighting targets diversity **within illicit cases**. Each feature is represented by its within-period empirical percentile relative to a fitting-only reference population, including unknown training transactions. Standardized confirmed illicit profiles form eight clusters. Smaller clusters receive greater initial weights, which are capped and normalized to preserve total illicit weight. A second weighting scheme clusters a balanced concatenation of feature percentiles and the 12 structural descriptors.

These clusters represent statistical profiles rather than named fraud types. The percentile representation determines sample weights; it is not appended to classifier inputs.


In [11]:
def period_percentiles(query, reference, self_in_reference=False):
    result = np.empty((len(query), 165), dtype=float)
    qtime = query.time_step.to_numpy()
    for period in np.unique(qtime):
        positions = np.flatnonzero(qtime == period)
        ref = reference.loc[reference.time_step.eq(period), FEATURES].to_numpy(float)
        values = query.iloc[positions][FEATURES].to_numpy(float)
        assert len(ref) > 1
        for j in range(165):
            ordered = np.sort(ref[:,j]); q = values[:,j]
            left = np.searchsorted(ordered,q,side='left'); right = np.searchsorted(ordered,q,side='right')
            result[positions,j] = (left+right-1)/(2*(len(ordered)-1)) if self_in_reference else (left+right)/(2*len(ordered))
    assert np.isfinite(result).all() and ((result>=0)&(result<=1)).all()
    return result

def subtype_weights(a, b=None, seed=47, stronger=False):
    reference = train_clean if b is None else train_clean.drop(index=confirmed.index[b])
    if b is not None: assert not reference.index.isin(confirmed.index[b]).any()
    positive = y[a] == 1
    context = period_percentiles(confirmed.iloc[a[positive]], reference, True)
    scaled = StandardScaler().fit_transform(context)
    graph = np.clip(StandardScaler().fit_transform(G[a[positive]]), -5, 5)
    representations = [scaled, np.column_stack([scaled/np.sqrt(165), graph/np.sqrt(12)])]
    weights = []
    for values in representations:
        with threadpool_limits(limits=1):
            groups = KMeans(n_clusters=8,n_init=10,random_state=seed).fit_predict(values)
        counts = np.bincount(groups,minlength=8)
        pw = np.clip(np.sqrt((positive.sum()/8)/counts[groups]), .5, 3.)
        pw /= pw.mean()
        if stronger:
            pw = np.clip(pw**2, .5, 3.)
            pw /= pw.mean()
        w = np.ones(len(a)); w[positive] = pw
        assert np.isclose(w[positive].sum(),positive.sum()) and (w[~positive]==1).all()
        weights.append(w)
    return weights

def fit_probability(family, a, fit_features, query_features, weights, seed, final=False):
    weights = weights.copy()
    if family == 'HGB':
        recency = 2.**(-(times[a].max()-times[a])/20.)
        recency /= recency.mean()
        weights *= recency
    prep = SimpleImputer(strategy='median',keep_empty_features=True).fit(fit_features)
    model = make_model(family,seed,final)
    with threadpool_limits(limits=2):
        model.fit(prep.transform(fit_features), y[a], sample_weight=weights/weights.mean())
        probability = model.predict_proba(prep.transform(query_features))[:,1]
    assert np.isfinite(probability).all() and ((probability>=0)&(probability<=1)).all()
    return probability

def predict_stage(name, a, query_features, b=None, seed=47, final=False):
    graph_inputs = name in ['HGB–CatBoost Graph Ensemble','HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting']
    fit_features = np.column_stack([X[a],G[a]]) if graph_inputs else X[a]
    if name in ['HGB','CatBoost','HGB + CatBoost']:
        families = [name] if name in ['HGB','CatBoost'] else ['HGB','CatBoost']
        return np.mean([fit_probability(family,a,fit_features,query_features,np.ones(len(a)),seed,final) for family in families],axis=0)
    weights = subtype_weights(a,b,seed,stronger=name=='HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting')
    schemes = [weights[0]] if name=='Feature-subtype blend' else weights
    predictions = []
    for w in schemes:
        predictions.append(.5*fit_probability('HGB',a,fit_features,query_features,w,seed,final)
            + .5*fit_probability('CatBoost',a,fit_features,query_features,w,seed,final))
    return predictions[0] if len(predictions)==1 else .75*predictions[0]+.25*predictions[1]

def evaluate_stage(name):
    rows = []
    for seed,fold,a,b in SPLITS:
        features = np.column_stack([X[b],G[b]]) if name in ['HGB–CatBoost Graph Ensemble','HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting'] else X[b]
        probability = predict_stage(name,a,features,b,seed)
        rows.append(metrics(name,seed,fold,y[b],probability))
        print(name,seed,fold,'complete')
    return rows


### 6.2 HGB

HGB models nonlinear thresholds and feature interactions. Its shallow trees, minimum leaf size 100 and L2 regularization constrain complexity.


In [12]:
hgb_results = show_stage('HGB')


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
HGB,9,0.998601,0.99784,0.992978,0.987153,0.948957,0.967673


**Result and interpretation.** Mean ROC-AUC is 0.998601. This improves on the linear baseline and establishes one of the two final component learners.


### 6.3 CatBoost

CatBoost provides a second boosting implementation with symmetric trees. The numeric-only schema is compatible with it; no categorical features are required.


In [13]:
cat_results = show_stage('CatBoost')


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
CatBoost,9,0.998451,0.99748,0.993127,0.992259,0.949231,0.970258


**Result and interpretation.** Mean ROC-AUC is 0.998451. It is weaker alone than HGB, but a component need not be the strongest standalone model to contribute useful blend predictions.


### 6.4 HGB and CatBoost probability blend

The two illicit probabilities are averaged with equal weights, using identical fitting data and validation splits. The comparison tests whether their predictions provide complementary information.


In [14]:
blend_results = show_stage('HGB + CatBoost')


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
HGB + CatBoost,9,0.998705,0.997921,0.99362,0.991216,0.949963,0.970147


**Result and interpretation.** The 50:50 blend reaches mean ROC-AUC 0.998705, above either standalone learner on this protocol. This is the foundation for the finalists; the component models are retained.


### 6.5 Weighting rare illicit feature profiles

Eight clusters of confirmed illicit feature-percentile profiles are fitted within each fold. Inverse-square-root cluster-frequency weights are normalized within illicit examples. This tests whether greater representation of minority illicit patterns improves classification without relabelling unknown rows.


In [15]:
feature_subtypes_results = show_stage('Feature-subtype blend')


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
Feature-subtype blend,9,0.998746,0.998155,0.993771,0.989827,0.952341,0.970716


**Result and interpretation.** Mean ROC-AUC rises to 0.998746. This feature-based weighting branch remains a component of both finalists.


### 6.6 Graph-informed subtype weighting

A second HGB/CatBoost pair uses subtype clusters that balance feature-profile and graph information. The resulting probabilities combine 75% of the feature-weighted pair with 25% of the graph-weighted pair. At this stage, the classifiers still receive 165 original inputs.


In [16]:
graph_weighted_results = show_stage('Graph-weighted blend')


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
Graph-weighted blend,9,0.998749,0.998121,0.993779,0.990199,0.951975,0.970704


**Result and interpretation.** Mean ROC-AUC is 0.998749, a small gain over feature-only subtype weighting. This weighting architecture forms part of both final strategies; the difference alone provides limited evidence of improvement.


### 6.7 Graph structural features: HGB–CatBoost Graph Ensemble

The four-learner recipe incorporates 12 structural descriptors alongside all 165 original numeric features. These descriptors expose topology directly to the classifiers.


In [17]:
graph_ensemble_results = show_stage('HGB–CatBoost Graph Ensemble')


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
HGB–CatBoost Graph Ensemble,9,0.998787,0.998084,0.993875,0.99068,0.952524,0.971222


**Result and interpretation.** Mean ROC-AUC is 0.998787, weakest-fold AUC 0.998084. Its reported Kaggle public score is 0.95132, the strongest confirmed public result among the retained finalists.


### 6.8 Stronger rare-subtype weighting: HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting

The HGB–CatBoost Graph Ensemble error audit found that rare profiles contained 9% of illicit examples but accounted for 66% of misses. HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting squares each existing illicit subtype weight, clips the result to 0.5-3, and renormalizes within illicit examples. The learners, graph features and blend fractions remain the same.


In [18]:
stronger_subtype_ensemble_results = show_stage('HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting')


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1
model,,,,,,,
HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting,9,0.998798,0.998111,0.993918,0.989753,0.953439,0.971247


**Result and interpretation.** Mean ROC-AUC is 0.998798, with six wins out of nine and worst-fold AUC 0.998111. Seed-47 rare-subtype recall rises from 64.9% to 67.4%. The gain is small and seed 49 worsens on average; its Kaggle score has not been reported.


## 7. Final model selection

**HGB–CatBoost Graph Ensemble** is selected as the strongest externally supported model, with the best confirmed Kaggle public ROC-AUC of 0.95132. **HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting** is selected as a targeted local candidate because it slightly improves nine-fold average and weakest-fold AUC and recovers more rare illicit cases.

These are two related recipes, not a new blend of the two CSV files. Each uses four learners on 177 inputs:

| Component | Probability weight |
|---|---:|
| HGB with feature-subtype weights | 37.5% |
| CatBoost with feature-subtype weights | 37.5% |
| HGB with graph-informed subtype weights | 12.5% |
| CatBoost with graph-informed subtype weights | 12.5% |

HGB additionally uses the 20-step recency weights. HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting changes only the subtype-weight transformation. All four learners are fitted to confirmed labels. All 165 original features remain included.


In [19]:
final_rows = [r for r in RECORDED_RESULTS if r['model'] in ['HGB–CatBoost Graph Ensemble','HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting']]
final_table = summarize(final_rows)
final_table['confirmed_Kaggle_public_AUC'] = [0.95132 if name=='HGB–CatBoost Graph Ensemble' else np.nan for name in final_table.index]
display(final_table)
paired = pd.DataFrame(final_rows).pivot(index=['seed','fold'],columns='model',values='auc')
paired['auc_change'] = paired['HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting']-paired['HGB–CatBoost Graph Ensemble']
display(paired.groupby(level='seed').mean())
print('HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting fold wins:', int((paired.auc_change>0).sum()), '/ 9')
print('Confirmation-only mean AUC change:', paired.drop(index=(47,1)).auc_change.mean())


,folds,mean_auc,worst_auc,mean_AP,precision,recall,F1,confirmed_Kaggle_public_AUC
model,,,,,,,,
HGB–CatBoost Graph Ensemble,9,0.998787,0.998084,0.993875,0.990680,0.952524,0.971222,0.95132
HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting,9,0.998798,0.998111,0.993918,0.989753,0.953439,0.971247,NaN


model,HGB–CatBoost Graph Ensemble,HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting,auc_change
seed,,,
47,0.998878,0.998891,0.000014
48,0.998672,0.998714,0.000043
49,0.998812,0.998790,-0.000022


HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting fold wins: 6 / 9
Confirmation-only mean AUC change: 1.0398088078678036e-05


**Selection rationale and limits.** HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting meets the recorded exploratory advancement rule: positive mean AUC change, positive mean on the eight non-screening folds, at least five fold wins, no paired decline below −0.00015, and weakest-fold AUC within 0.00005 of the benchmark's weakest fold. It wins 6/9 folds. However, these are reused random folds, not an independent test, and the seed-49 average change is negative. The roughly 0.000011 mean AUC gain does not establish a higher Kaggle score.

Precision, recall and F1 below use threshold 0.5; changing that threshold cannot improve ROC-AUC. Temporal consistency remains unverified by this random-validation protocol. Kaggle's public result also does not reveal private-test performance.


### 7.1 Residual error analysis

The seed-47 out-of-fold probabilities provide one held-out prediction per confirmed transaction for a transaction-level comparison. This three-fold diagnostic is separate from the nine-fold averages above. Rare-subtype assignments are fitted using each fold's training positives. Cluster membership is summarized by training support rather than interpreted as named fraud types.


In [20]:
diagnostics = []
rare_illicit = (FINAL_OOF['subtype_size16_seed47'] < 200) & (y == 1)
for name,key in [('HGB–CatBoost Graph Ensemble','p16_seed47'),('HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting','p19_seed47')]:
    probability = FINAL_OOF[key]; predicted = probability >= .5
    tn,fp,fn,tp = confusion_matrix(y,predicted,labels=[0,1]).ravel()
    diagnostics.append(dict(model=name, pooled_AUC=roc_auc_score(y,probability),
        precision=precision_score(y,predicted), recall=recall_score(y,predicted),
        F1=f1_score(y,predicted), TP=tp,FN=fn,FP=fp,TN=tn,
        rare_illicit=int(rare_illicit.sum()),rare_misses=int((~predicted[rare_illicit]).sum()),
        rare_recall=float(predicted[rare_illicit].mean())))
display(pd.DataFrame(diagnostics).set_index('model'))
roles = np.full(len(y),'other',dtype=object)
ki = indegree[known_nodes]; ko = outdegree[known_nodes]
roles[(ki>=2)&(ki>=2*ko)] = 'consolidation-like'
roles[(ko>=2)&(ko>=2*ki)] = 'distribution-like'
p16 = FINAL_OOF['p16_seed47']; missed16 = (y==1)&(p16<.5)
role_rows = []
for role in ['consolidation-like','distribution-like','other']:
    m = (roles==role)&(y==1)
    role_rows.append(dict(role=role,illicit=int(m.sum()),missed=int(missed16[m].sum()),recall=float((p16[m]>=.5).mean())))
display(pd.DataFrame(role_rows))


,pooled_AUC,precision,recall,F1,TP,FN,FP,TN,rare_illicit,rare_misses,rare_recall
model,,,,,,,,,,,
HGB–CatBoost Graph Ensemble,0.998867,0.990014,0.952250,0.970765,3470,174,35,27556,328,115,0.64939
HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting,0.998875,0.989471,0.954171,0.971500,3477,167,37,27554,328,107,0.67378


,role,illicit,missed,recall
0,consolidation-like,116,36,0.689655
1,distribution-like,218,26,0.880734
2,other,3310,112,0.966163


**Interpretation.** HGB–CatBoost Graph Ensemble misses 174 illicit cases and raises 35 false alarms in this pass. HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting misses 167 and raises 37. Rare-subtype misses fall from 115 to 107; this is a useful but incomplete improvement. Consolidation-like nodes have low recall, but 35 of their 36 misses also belong to the rare-subtype group, so these findings overlap. Unknown labels remain unknown; there is no ground truth for measuring their classification accuracy. The residual error pattern supports retaining subtype weighting while documenting its limits.


### 7.2 Final prediction files

The final models were retrained on all 31,235 confirmed training transactions. Unknown training covariates define the label-free period reference, and the supplied graph defines the 12 structural features.

The **HGB–CatBoost Graph Ensemble** produces `submission_16.csv`; the **HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting** produces `submission_19.csv`. Both files contain illicit probabilities, and their format and probability ranges are validated below.


In [21]:
def validate_submission(filename):
    result = pd.read_csv(filename)
    assert result.columns.tolist() == sample.columns.tolist() and len(result) == len(sample)
    key,target = sample.columns
    assert result[key].equals(sample[key])
    assert result[target].notna().all() and result[target].between(0,1).all()
    return dict(file=filename,rows=len(result),sha256=hashlib.sha256(Path(filename).read_bytes()).hexdigest())

def rebuild_submission(name,filename):
    a = np.arange(len(y))
    query = np.column_stack([test_clean[FEATURES].to_numpy(float),G_test])
    probability = predict_stage(name,a,query,seed=47,final=True)
    key,target = sample.columns
    assert test_clean[key].is_unique
    lookup = pd.Series(probability,index=test_clean[key])
    result = sample.copy(); result[target] = result[key].map(lookup)
    assert result[target].notna().all() and result[target].between(0,1).all()
    payload = result.to_csv(index=False).encode('utf-8')
    output = Path(filename)
    if output.exists():
        if output.read_bytes() != payload:
            raise FileExistsError(f'{filename} is preserved: regenerated bytes differ. Review the comparison before replacement.')
    else:
        output.write_bytes(payload)
    return validate_submission(filename)

if REBUILD_SUBMISSIONS:
    for name,filename in [('HGB–CatBoost Graph Ensemble','submission_16.csv'),('HGB–CatBoost Graph Ensemble with Stronger Subtype Weighting','submission_19.csv')]:
        rebuild_submission(name,filename)
display(pd.DataFrame([validate_submission('submission_16.csv'),validate_submission('submission_19.csv')])[['file','rows']])


,file,rows
0,submission_16.csv,15329
1,submission_19.csv,15329
